# Tier 2: run penuh di GPU Colab

Notebook ini hanya peluncur: semua logika (data, model, pretraining, fine-tuning, evaluasi) ada di
modul `src/fraud/training/tier2/` dan dipanggil lewat `uv run python -m ...`. Semua run tercatat ke
MLflow di mesin pengembangan lewat tunnel, sama dengan eksperimen tier 1.

**Prasyarat di mesin pengembangan, selama sesi berjalan:**

```zsh
cd ~/Projects/realtime-fraud-detection-platform
docker compose up -d postgres mlflow
caffeinate -i -d &
# Basic auth dipasang lewat traffic policy ngrok yang disimpan di luar repo.
ngrok http 5001 --traffic-policy-file ~/ngrok-mlflow-policy.yml
```

Matikan tunnel begitu sesi selesai. Mesin harus tetap menyala dan terhubung internet.

**Colab Secrets** (ikon kunci di panel kiri, aktifkan akses notebook):

| Secret | Isi |
|---|---|
| `MLFLOW_TRACKING_URI` | URL HTTPS dari tunnel |
| `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` | Wajib, sama dengan kredensial di policy |
| `PARQUET_DRIVE_DIR` | Path folder Parquet di bawah `MyDrive`, berisi partisi `txn_day=N/` |

**Urutan:** sel 1 sampai 5 di setiap sesi baru. Sel 6 cukup sekali. Sel 7 sampai 10 bisa diulang
setelah sesi terputus; run yang sudah selesai dilewati otomatis. Evaluasi split uji sengaja tidak
ada di sini dan dijalankan terpisah, karena hanya boleh sekali.

## 1. GPU dan driver

In [ ]:
!nvidia-smi

## 2. Kode dari GitHub

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/daffaalhanif/realtime-fraud-detection-platform.git"
REPO_DIR = "/content/realtime-fraud-detection-platform"


def run_python(snippet: str) -> str:
    """Menjalankan potongan Python di environment repo dan mengembalikan output-nya."""
    result = subprocess.run(["uv", "run", "python", "-c", snippet], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-3000:])
    return result.stdout


if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="commit %h %s"

## 3. Dependency dan CUDA

In [ ]:
%pip install -q uv
# Grup notebook (ipykernel) tidak dibutuhkan; Colab punya kernel sendiri.
!uv sync --no-group notebook

print(run_python(
    "import torch\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda)\n"
    "if not torch.cuda.is_available():\n"
    "    raise SystemExit('PyTorch tidak melihat GPU; periksa driver di output sel 1.')\n"
    "print('GPU', torch.cuda.get_device_name(0))"
))

## 4. Data dari Google Drive

Disalin ke disk lokal Colab, ke path bawaan modul. Membaca ratusan partisi langsung dari Drive jauh
lebih lambat. Jumlah partisi dan baris harus sama dengan data di mesin pengembangan.

In [ ]:
# Paket ini hanya ada di runtime Colab, tidak di environment lokal.
from google.colab import drive, userdata  # pyright: ignore[reportMissingImports]

drive.mount("/content/drive")
source = os.path.join("/content/drive/MyDrive", userdata.get("PARQUET_DRIVE_DIR"))
target = os.path.join(REPO_DIR, "data/processed/transactions")
if not os.path.isdir(target):
    os.makedirs(os.path.dirname(target), exist_ok=True)
    !cp -r "{source}" "{target}"

partitions = [name for name in os.listdir(target) if name.startswith("txn_day=")]
print(len(partitions), "partisi")
print(run_python(
    f"import pandas as pd; print(len(pd.read_parquet({target!r}, columns=['TransactionID'])))"
).strip(), "baris")

## 5. Koneksi MLflow

In [ ]:
# Ketiganya wajib: tunnel selalu memakai basic auth, jadi secret yang kosong harus gagal di sini,
# bukan muncul belakangan sebagai penolakan koneksi MLflow.
for name in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    os.environ[name] = userdata.get(name)
os.environ["PYTHONUNBUFFERED"] = "1"

print(run_python(
    "from fraud.training.tier2.tracking import EXPERIMENT_NAME, setup_mlflow\n"
    "print(EXPERIMENT_NAME, 'id', setup_mlflow(False))"
))

## 6. Smoke di GPU (sekali)

Membuktikan jalur CUDA dan mixed precision, yang tidak bisa diuji di mesin pengembangan. Tercatat di
eksperimen smoke.

In [ ]:
!uv run python -m fraud.training.tier2.finetune pipeline --arm ordered --seq-len 16 --smoke

## 7. Pemilihan konfigurasi (bisa dilanjutkan)

In [ ]:
!uv run python -m fraud.training.tier2.finetune select --dry-run

In [ ]:
!uv run python -m fraud.training.tier2.finetune select

## 8. Konfigurasi terpilih

In [ ]:
!uv run python -m fraud.training.tier2.finetune select-report

## 9. Run utama ketiga lengan (bisa dilanjutkan)

In [ ]:
!uv run python -m fraud.training.tier2.ablation_shuffle --dry-run

In [ ]:
!uv run python -m fraud.training.tier2.ablation_shuffle

## 10. Evaluasi validasi

Diagnostik dan putusan promosi T1+. Boleh juga dijalankan di mesin pengembangan.

In [ ]:
!uv run python -m fraud.training.tier2.evaluate --split validation